# Por que os dados sintéticos foram descartados — evidência de três testes

**O que este notebook registra.** Depois de gerar runs sintéticas com K-means
(`geracao_dados_sinteticos_kmeans.ipynb`) e medir que elas pioravam o modelo, a
pergunta ficou: isso é específico daquele método, ou é um limite estrutural do
problema? Este notebook responde com três testes independentes.

**O resultado, adiantado:** os três convergem. Dado derivado das 7 features
existentes não melhora o modelo, com nenhuma variante de geração testada — e
duas pessoas do grupo chegaram nisso por caminhos diferentes.

**O que este notebook NÃO faz:** não altera nenhum dos três notebooks de modelo
publicados (`Decision_tree_regre.ipynb`, `Grad_Boosting_Regre.ipynb`,
`Random_for_regre.ipynb`). É só o registro da investigação.

## Uma advertência sobre metodologia, antes dos números

A primeira rodada desta investigação (fora deste notebook) produziu um "ganho"
de 2,2 °C que era **vazamento**: o sintético era gerado uma vez, fora do laço
de validação, então cópias de uma run real vazavam para o treino do fold em
que essa mesma run estava no teste. Os três testes abaixo geram o sintético
**dentro de cada fold**, usando só as linhas de treino daquele fold — sem
exceção. Essa é a diferença entre medir o método e medir o vazamento.


## Teste 1 — dataset agregado, features enriquecidas com `reason`

Reaproveita a extração de 9 fatores escondidos no texto (`run_id` + `reason`)
feita em paralelo a este notebook: velocidade do soprador, presença de
cooktop, variante de bandeja, duto, defletor, etc. Essa extração já mostrou,
sozinha, ganho real (RMSE 8,084 → 7,548 °C; piso de ruído 7,41 → 4,09 °C).

A pergunta aqui é outra: **dado esse espaço de features melhor, gerar linhas
sintéticas adicionais ajuda ainda mais?**


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, re
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score

RANDOM_STATE, N_TREES = 0, 150
FATORES = ['blower_direction', 'housing', 'board_position', 'run_case',
           'losses', 'deflector_hole', 'material_coil_support']
ALVO, GRUPO = 'temp', 'run_id'
DESCONHECIDO = 'desconhecido'

df = pd.read_csv('../data/dataset_pre_processado.csv')
df.columns = [c.strip().replace('﻿', '') for c in df.columns]

por_run = df.groupby(GRUPO)[FATORES].first()
reason_por_run = df.groupby(GRUPO)['reason'].first()

# O texto vem de DUAS fontes: o proprio run_id (ex.: "Run 3 - Pyro Fans in Low") e a
# coluna reason. O diagnostico anterior minerou so o reason e perdeu metade do sinal.
TEXTO = (por_run.index.to_series().astype(str) + ' | ' + reason_por_run.astype(str)).str.lower()


def blower_setting(s):
    """Ajuste do soprador do IPC. NAO confundir com os fans do forno (ver fan_setting)."""
    if re.search(r'16\s*cfm', s):      return '16cfm'
    if 'double flow' in s:             return 'double'
    if 'blower on high' in s:          return 'high'
    if 'blower on medium' in s:        return 'medium'
    if 'blower on' in s:               return 'on'
    return DESCONHECIDO


def fan_setting(s):
    """Fans do forno. 'Fan velocity 1500 RPM' e 'Fans in Low' descrevem o mesmo ensaio."""
    if 'fans in low' in s or 'fan on low' in s or '1500 rpm' in s:
        return 'low'
    return DESCONHECIDO


def tray(s):
    if 'simplified tray' in s: return 'simplified'
    if 'hole tray' in s:       return 'hole'
    if 'real tray' in s:       return 'real'
    return DESCONHECIDO


def duct(s):
    if 'top duct' in s:      return 'top'
    if 'mainback duct' in s: return 'mainback'
    if 'new duct' in s:      return 'new'
    return DESCONHECIDO


def deflector_var(s):
    if 'cmc deflector' in s:      return 'cmc'
    if 'new deflector' in s:      return 'new'
    if 'new hole position' in s:  return 'hole_pos'
    if 'no back hole' in s:       return 'no_back'
    return DESCONHECIDO


def cover_hs(s):
    # "No cover on HS" e informacao positiva de ausencia; a maioria das runs nao diz nada.
    if 'no cover' in s: return 'nao'
    if 'cover' in s:    return 'sim'
    return DESCONHECIDO


extras = pd.DataFrame({
    'blower_setting': TEXTO.map(blower_setting),
    'fan_setting':    TEXTO.map(fan_setting),
    'tray':           TEXTO.map(tray),
    'duct':           TEXTO.map(duct),
    'deflector_var':  TEXTO.map(deflector_var),
    'cover_hs':       TEXTO.map(cover_hs),
    # Estas duas sao mencao explicita de ausencia/presenca: o default e seguro.
    'cooktop':        np.where(TEXTO.str.contains('wo cookitop|wo cooktop'), 'nao', 'sim'),
    'spacer':         np.where(TEXTO.str.contains('spacer'), 'sim', 'nao'),
    'igbt_order':     np.where(TEXTO.str.contains('diff order igbt'), 'alternada', 'padrao'),
}, index=por_run.index)

EXTRAS = list(extras.columns)

print('=== cobertura por variavel ===')
cob = pd.DataFrame({
    'niveis': [extras[c].nunique() for c in EXTRAS],
    'runs com valor conhecido': [int((extras[c] != DESCONHECIDO).sum()) for c in EXTRAS],
    'valores': [' / '.join('%s:%d' % (k, v) for k, v in extras[c].value_counts().items())
                for c in EXTRAS],
}, index=EXTRAS)
print(cob.to_string())

chave_velha = por_run.astype(str).agg(' | '.join, axis=1)
chave_nova = (chave_velha + ' | ' + extras.astype(str).agg('|'.join, axis=1))
print()
print('configuracoes distintas -- so as 7 colunas : %d / 56' % chave_velha.nunique())
print('configuracoes distintas -- com os extras   : %d / 56' % chave_nova.nunique())

In [ ]:
# ---- Recalculo do piso de ruido ----
# O piso e a discordancia entre runs que o modelo NAO consegue distinguir. Ao codificar
# variaveis que antes estavam so no texto, runs que eram "replicas contraditorias" viram
# casos distintos -- e a discordancia delas sai do piso.

df['combo_velho'] = df[GRUPO].map(chave_velha)
df['combo_novo'] = df[GRUPO].map(chave_nova)


def piso_de_ruido(coluna_combo):
    est = df.groupby([coluna_combo, 'tc_number'])[ALVO].agg(['count', 'std'])
    est = est[est['count'] > 1]
    return float(np.sqrt((est['std'] ** 2).mean())), len(est)


piso_velho, n_velho = piso_de_ruido('combo_velho')
piso_novo, n_novo = piso_de_ruido('combo_novo')

print('%-34s %8s %10s' % ('', 'piso (C)', 'pares'))
print('%-34s %8.2f %10d' % ('features atuais (7 colunas)', piso_velho, n_velho))
print('%-34s %8.2f %10d' % ('features + extras do texto', piso_novo, n_novo))
print()
print('queda do piso: %.2f C (%.0f%%)' % (piso_velho - piso_novo,
                                          100 * (piso_velho - piso_novo) / piso_velho))
print()
print('ATENCAO: o piso novo e estimado sobre %d pares em vez de %d. Menos pares = estimativa'
      % (n_novo, n_velho))
print('mais ruidosa. A queda e real, mas a incerteza sobre o NOVO valor e maior.')

### Três variantes de geração, pré-registradas

- **A** — configuração inédita + interpolação convexa entre duas runs-mãe do
  mesmo cluster térmico (o método original, agora no espaço enriquecido).
- **B** — réplica de uma run real + ruído de medição (augmentation clássica).
- **C** — perturbação na direção de um vizinho que difere em exatamente 1 fator.

Todas reportadas — não só a que desse melhor resultado. Rodar várias
variantes e reportar só a vencedora é como se fabrica uma melhora falsa.


In [ ]:
# ---- Fase 2: tres variantes de geracao, no espaco enriquecido ----
# Pre-registradas antes de rodar. TODAS sao reportadas, nao so a melhor.
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import silhouette_score

SIGMA = 2.18            # desvio mediano entre replicas reais
N_SINT = 44
rng_global = np.random.default_rng(RANDOM_STATE)

TC_INFO = df.groupby('tc_number')[['tc_name', 'key_area', 'active_limit']].first()
piv = df.pivot_table(index=GRUPO, columns='tc_number', values=ALVO, aggfunc='mean')
piv_imp = piv.fillna(piv.median())
Z = (piv_imp - piv_imp.mean()) / piv_imp.std().replace(0, 1.0)

TODOS_FATORES = FATORES + EXTRAS
cfg_por_run = pd.concat([por_run, extras], axis=1)[TODOS_FATORES]


def montar_linhas(run_id, cfg, perfil):
    """Transforma um perfil (Series tc_number -> temp) em linhas no schema do dataset."""
    out = []
    for tc, t in perfil.dropna().items():
        info = TC_INFO.loc[tc]
        linha = {GRUPO: run_id, 'tc_number': tc, 'tc_name': info['tc_name'],
                 'key_area': info['key_area'], 'active_limit': info['active_limit'],
                 ALVO: round(float(t), 2), 'reason': 'SINTETICO'}
        linha.update({c: cfg[c] for c in TODOS_FATORES})
        out.append(linha)
    return out


# ---------- Variante A: configuracoes ineditas + interpolacao por cluster ----------
def variante_A(seed):
    rng = np.random.default_rng(seed)
    km = KMeans(n_clusters=3, n_init=25, random_state=RANDOM_STATE).fit(Z)
    rot = pd.Series(km.labels_, index=Z.index)
    Xf = pd.get_dummies(cfg_por_run.astype(str)).astype(float)
    clf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
    clf.fit(Xf.values, rot.values)
    membros = {c: rot.index[rot == c].tolist() for c in np.unique(rot)}

    combos_reais = set(cfg_por_run.astype(str).agg(' | '.join, axis=1))

    # Sorteio condicionado, como na primeira rodada: sortear os 16 fatores de forma
    # independente geraria hardware impossivel (Pyro so existe com No losses, etc).
    # run_case ancora tudo; os extras vem em bloco de uma run real do mesmo run_case,
    # preservando a co-ocorrencia realista entre eles.
    p_rc = cfg_por_run['run_case'].value_counts(normalize=True)
    cond = {rc: sub for rc, sub in cfg_por_run.groupby('run_case')}
    p_hous = cfg_por_run['housing'].value_counts(normalize=True)
    p_defl = {h: s['deflector_hole'].value_counts(normalize=True)
              for h, s in cfg_por_run.groupby('housing')}

    def sortear_cfg():
        rc = rng.choice(p_rc.index.to_numpy(), p=p_rc.to_numpy())
        sub = cond[rc]
        hs = rng.choice(p_hous.index.to_numpy(), p=p_hous.to_numpy())
        pd_ = p_defl[hs]
        cfg = {'run_case': rc,
               'losses': sub['losses'].sample(1, random_state=int(rng.integers(1e9))).iloc[0],
               'blower_direction': sub['blower_direction'].sample(1, random_state=int(rng.integers(1e9))).iloc[0],
               'housing': hs,
               'deflector_hole': rng.choice(pd_.index.to_numpy(), p=pd_.to_numpy()),
               'board_position': cfg_por_run['board_position'].sample(1, random_state=int(rng.integers(1e9))).iloc[0],
               'material_coil_support': cfg_por_run['material_coil_support'].sample(1, random_state=int(rng.integers(1e9))).iloc[0]}
        doador = sub.sample(1, random_state=int(rng.integers(1e9))).iloc[0]
        cfg.update({c: doador[c] for c in EXTRAS})
        return cfg

    linhas, vistos, tent = [], set(combos_reais), 0
    while len({l[GRUPO] for l in linhas}) < N_SINT and tent < 50000:
        tent += 1
        cfg = sortear_cfg()
        ch = ' | '.join(str(cfg[c]) for c in TODOS_FATORES)
        if ch in vistos:
            continue
        vistos.add(ch)
        x = pd.get_dummies(pd.DataFrame([cfg])[TODOS_FATORES].astype(str))
        x = x.reindex(columns=Xf.columns, fill_value=0.0).astype(float)
        p = clf.predict_proba(x.values)[0]
        c = int(clf.classes_[rng.choice(len(clf.classes_), p=p)])
        a, b = rng.choice(membros[c], size=2, replace=False)
        lam = rng.uniform(0.2, 0.8)
        perfil = lam * piv.loc[a] + (1 - lam) * piv.loc[b]
        perfil = perfil + rng.normal(0, SIGMA, len(perfil))
        linhas += montar_linhas('SYN-A-%03d' % len({l[GRUPO] for l in linhas}), cfg, perfil)
    return pd.DataFrame(linhas)


# ---------- Variante B: mesma configuracao real + ruido de medicao ----------
def variante_B(seed):
    rng = np.random.default_rng(seed)
    linhas = []
    escolhidas = rng.choice(piv.index.to_numpy(), size=N_SINT, replace=True)
    for i, mae in enumerate(escolhidas):
        cfg = cfg_por_run.loc[mae].to_dict()
        perfil = piv.loc[mae] + rng.normal(0, SIGMA, piv.shape[1])
        linhas += montar_linhas('SYN-B-%03d' % i, cfg, perfil)
    return pd.DataFrame(linhas)


# ---------- Variante C: perturbacao na direcao de um vizinho a 1 fator ----------
def variante_C(seed):
    rng = np.random.default_rng(seed)
    chaves = cfg_por_run.astype(str)
    vizinhos = {}
    for r in chaves.index:
        d = (chaves != chaves.loc[r]).sum(axis=1)
        v = d[(d == 1)].index.tolist()
        if v:
            vizinhos[r] = v
    if not vizinhos:
        return None
    linhas, mais = [], list(vizinhos.keys())
    for i in range(N_SINT):
        mae = mais[rng.integers(len(mais))]
        viz = vizinhos[mae][rng.integers(len(vizinhos[mae]))]
        cfg = cfg_por_run.loc[mae].to_dict()          # herda a config da mae, nao do vizinho
        lam = rng.uniform(0.75, 0.95)                  # fica perto da mae
        perfil = lam * piv.loc[mae] + (1 - lam) * piv.loc[viz]
        perfil = perfil + rng.normal(0, SIGMA, len(perfil))
        linhas += montar_linhas('SYN-C-%03d' % i, cfg, perfil)
    return pd.DataFrame(linhas)


print('pares de runs que diferem em exatamente 1 fator:',
      sum(1 for r in cfg_por_run.index
          for _ in [(cfg_por_run.astype(str) != cfg_por_run.astype(str).loc[r]).sum(axis=1)]) and
      int(((cfg_por_run.astype(str).apply(lambda row: (cfg_por_run.astype(str) != row).sum(axis=1), axis=1)) == 1).sum().sum() / 2))

In [ ]:
# ---- Preparo do dataset agregado para a ablacao (usado pela Secao seguinte) ----
# So o setup: X_de(), df_enriq, X_real, y, g. A ablacao em si -- e o harness
# leak-free -- esta na proxima celula.
COLS_X = TODOS_FATORES + ['key_area', 'tc_name']


def X_de(dados):
    X = pd.get_dummies(dados[COLS_X].astype(str))
    X['active_limit'] = dados['active_limit'].values
    X['tc_number'] = dados['tc_number'].values
    return X.astype(float)


df_enriq = df.copy()
for c in EXTRAS:
    df_enriq[c] = df_enriq[GRUPO].map(extras[c])

y, g = df_enriq[ALVO].values, df_enriq[GRUPO].values
X_real = X_de(df_enriq)

print('dataset preparado: %d linhas, %d colunas em X' % (len(df_enriq), X_real.shape[1]))

In [ ]:
# ---- Harness corrigido: geracao DENTRO de cada fold ----
#
# O erro da primeira tentativa: gerar o sintetico uma vez, fora do laco de validacao.
# As runs-mae vazavam do teste para o treino atraves das copias sinteticas, e o modelo
# era testado em runs cujo perfil ele ja tinha visto (quase identico) no treino.
# Isso produziu um "ganho" de 2.2 C que era puro vazamento.
#
# Protocolo correto: em cada fold, o gerador so enxerga as runs de TREINO daquele fold.
# KMeans, classificador e runs-mae -- tudo restrito ao treino.

from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier

SIGMA, N_SINT = 2.18, 20   # reduzido de 44 para 20: mais rapido, mesma conclusao qualitativa


def gerar_A(rng, permitidas):
    """Configuracoes ineditas + interpolacao por cluster. So runs de treino."""
    piv_t = piv.loc[list(permitidas)]
    Zt = piv_t.fillna(piv_t.median())
    Zt = (Zt - Zt.mean()) / Zt.std().replace(0, 1.0)
    k = min(3, max(2, len(permitidas) // 8))
    rot = pd.Series(KMeans(n_clusters=k, n_init=10, random_state=0).fit(Zt).labels_, index=Zt.index)
    cfg_t = cfg_por_run.loc[list(permitidas)]
    Xf = pd.get_dummies(cfg_t.astype(str)).astype(float)
    clf = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(Xf.values, rot.values)
    membros = {c: rot.index[rot == c].tolist() for c in np.unique(rot)}
    membros = {c: m for c, m in membros.items() if len(m) >= 2}
    if not membros:
        return None

    p_rc = cfg_t['run_case'].value_counts(normalize=True)
    cond = {rc: s for rc, s in cfg_t.groupby('run_case')}
    p_h = cfg_t['housing'].value_counts(normalize=True)
    p_d = {h: s['deflector_hole'].value_counts(normalize=True) for h, s in cfg_t.groupby('housing')}
    reais = set(cfg_t.astype(str).agg(' | '.join, axis=1))

    linhas, vistos, n, tent = [], set(reais), 0, 0
    while n < N_SINT and tent < 20000:
        tent += 1
        rc = rng.choice(p_rc.index.to_numpy(), p=p_rc.to_numpy())
        sub = cond[rc]
        hs = rng.choice(p_h.index.to_numpy(), p=p_h.to_numpy())
        doador = sub.iloc[rng.integers(len(sub))]
        cfg = {'run_case': rc,
               'losses': sub['losses'].iloc[rng.integers(len(sub))],
               'blower_direction': sub['blower_direction'].iloc[rng.integers(len(sub))],
               'housing': hs,
               'deflector_hole': rng.choice(p_d[hs].index.to_numpy(), p=p_d[hs].to_numpy()),
               'board_position': cfg_t['board_position'].iloc[rng.integers(len(cfg_t))],
               'material_coil_support': cfg_t['material_coil_support'].iloc[rng.integers(len(cfg_t))]}
        cfg.update({c: doador[c] for c in EXTRAS})
        ch = ' | '.join(str(cfg[c]) for c in TODOS_FATORES)
        if ch in vistos:
            continue
        vistos.add(ch)
        x = pd.get_dummies(pd.DataFrame([cfg])[TODOS_FATORES].astype(str))
        x = x.reindex(columns=Xf.columns, fill_value=0.0).astype(float)
        p = clf.predict_proba(x.values)[0]
        cands = [c for c in clf.classes_ if c in membros]
        pr = np.array([p[list(clf.classes_).index(c)] for c in cands], float)
        pr = pr / pr.sum() if pr.sum() > 0 else np.ones(len(cands)) / len(cands)
        c = cands[rng.choice(len(cands), p=pr)]
        a, b = rng.choice(membros[c], size=2, replace=False)
        lam = rng.uniform(0.2, 0.8)
        perfil = lam * piv.loc[a] + (1 - lam) * piv.loc[b]
        linhas += montar_linhas('SYN-A-%03d' % n, cfg, perfil + rng.normal(0, SIGMA, len(perfil)))
        n += 1
    return pd.DataFrame(linhas) if linhas else None


def gerar_B(rng, permitidas):
    """Replica de run real + ruido de medicao. So runs de treino."""
    perm = list(permitidas)
    linhas = []
    for i in range(N_SINT):
        mae = perm[rng.integers(len(perm))]
        perfil = piv.loc[mae] + rng.normal(0, SIGMA, piv.shape[1])
        linhas += montar_linhas('SYN-B-%03d' % i, cfg_por_run.loc[mae].to_dict(), perfil)
    return pd.DataFrame(linhas)


def gerar_C(rng, permitidas):
    """Perturbacao rumo a um vizinho a 1 fator. Mae E vizinho so do treino."""
    ch = cfg_por_run.loc[list(permitidas)].astype(str)
    viz = {}
    for r in ch.index:
        d = (ch != ch.loc[r]).sum(axis=1)
        v = d[d == 1].index.tolist()
        if v:
            viz[r] = v
    if not viz:
        return None
    maes = list(viz)
    linhas = []
    for i in range(N_SINT):
        m = maes[rng.integers(len(maes))]
        v = viz[m][rng.integers(len(viz[m]))]
        lam = rng.uniform(0.75, 0.95)
        perfil = lam * piv.loc[m] + (1 - lam) * piv.loc[v]
        linhas += montar_linhas('SYN-C-%03d' % i, cfg_por_run.loc[m].to_dict(),
                                perfil + rng.normal(0, SIGMA, len(perfil)))
    return pd.DataFrame(linhas)


def ablacao_correta(gerador, seed):
    """Gera dentro de cada fold, usando SO as runs de treino daquele fold."""
    rng = np.random.default_rng(seed)
    preds = np.zeros(len(df_enriq))
    for tr, te in GroupKFold(5).split(X_real, y, groups=g):
        runs_treino = sorted(pd.unique(g[tr]))   # ordem estavel: set de strings varia por processo
        Xtr, ytr = X_real.iloc[tr], y[tr]
        if gerador is not None:
            ds = gerador(rng, runs_treino)
            if ds is not None:
                Xs = X_de(ds)
                Xtr2, Xs = Xtr.align(Xs, join='outer', axis=1, fill_value=0.0)
                Xte = X_real.iloc[te].reindex(columns=Xtr2.columns, fill_value=0.0)
                Xtr = pd.concat([Xtr2, Xs], ignore_index=True)
                ytr = np.concatenate([ytr, ds[ALVO].values])
            else:
                Xte = X_real.iloc[te]
        else:
            Xte = X_real.iloc[te]
        m = RandomForestRegressor(n_estimators=N_TREES, random_state=seed, n_jobs=-1)
        m.fit(Xtr.values, ytr)
        preds[te] = m.predict(Xte.values)
    return float(np.sqrt(((preds - y) ** 2).mean()))

### Resultado — Teste 1

**Nenhuma variante melhora o modelo.** Só a A (configurações inéditas + cluster) piora com clareza
(+1,1 °C, bem acima do desvio entre sementes). A réplica com ruído (B, +0,04) é neutra, e a
perturbação rumo a vizinho (C, +0,20) fica a cerca de um desvio entre sementes do zero — não dá para
distinguir de ruído. Com 5 sementes e geração determinística, o resultado se reproduz entre execuções.


In [ ]:
# ---- Resumo do Teste 1 (dataset agregado) ----
SEEDS = [0, 1, 2, 3, 4]
geradores_1 = {'A: configuracoes ineditas + cluster': gerar_A,
               'B: replica + ruido de medicao': gerar_B,
               'C: perturbacao rumo a vizinho': gerar_C}

linhas_1 = []
base_1 = [ablacao_correta(None, s) for s in SEEDS]
linhas_1.append({'variante': 'so real (sem sintetico)', 'RMSE': np.mean(base_1),
                 'desvio entre sementes': np.std(base_1), 'delta': 0.0})
for nome, fn in geradores_1.items():
    vals = [ablacao_correta(fn, s) for s in SEEDS]
    linhas_1.append({'variante': nome, 'RMSE': np.mean(vals),
                     'desvio entre sementes': np.std(vals),
                     'delta': np.mean(vals) - np.mean(base_1)})

teste1_resumo = pd.DataFrame(linhas_1).set_index('variante').round(3)
print('=== TESTE 1: dataset agregado, features enriquecidas, harness leak-free ===')
print(teste1_resumo.to_string())

## Teste 2 — a mesma pergunta, na metodologia real dos modelos publicados

O Teste 1 usa um RandomForest genérico como régua de medição, no dataset
agregado (uma linha por termopar). Os três notebooks de modelo publicados
(`Random_for_regre.ipynb` etc.) usam outra estrutura: **formato wide por
`key_area`**, uma linha por run, cada termopar é uma coluna-alvo
(multi-output), 7 variáveis ordinais, e validação `LeaveOneGroupOut`.

Esse teste replica a codificação e o pré-processamento exatamente — mesma
codificação ordinal, mesmas correções de grafia (`GRAFIAS_DIVERGENTES`), mesmo
filtro de cobertura mínima — e injeta sintético gerado **dentro de cada fold**,
para responder à pergunta com a régua que o time realmente usa.

Duas simplificações assumidas, ambas por custo computacional:

- **`GroupKFold(5)` em vez de `LeaveOneGroupOut`.** O `LeaveOneGroupOut` real
  (~50 folds por área) combinado com geração de sintético a cada fold — K-means
  + classificador + regressor, repetido por área × semente × variante — travou
  a primeira tentativa de execução. `GroupKFold(5)` é a mesma ideia (nenhuma
  configuração de teste aparece no treino), com 10x menos folds. É uma validação
  menos exaustiva, não menos honesta: o requisito que importa para esta pergunta
  — treino e teste nunca compartilham a mesma run — continua garantido.
- **Hiperparâmetros fixos** (`n_estimators=150`) em vez do `GridSearchCV` por
  área que `Random_for_regre.ipynb` faz — repetir a busca em grade dentro de
  cada fold de cada área seria caro demais para esta checagem, e o que se está
  testando é o efeito do dado, não do tuning.


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

RANDOM_STATE = 42          # mesma SEMENTE do Random_for_regre.ipynb
N_TREES = 150               # fixo (nao replica o GridSearch por area, ver ressalva)
SIGMA = 2.18
COBERTURA_MINIMA_ALVO = 0.50
N_FOLDS = 5                 # GroupKFold em vez de LeaveOneGroupOut: LOO com ~50 folds
                             # por area x 7 areas x sementes x variantes ficou caro
                             # demais (ver ressalva de metodologia na Secao 4).

AREAS = ['mainback_data.csv', 'coil_temps_data.csv', 'main_vent_sys_data.csv',
         'ct_knobs_data.csv', 'ipc_comp_data.csv', 'ipc_cooling_data.csv',
         'oven_hmi_data.csv']

VARIAVEIS_INDEPENDENTES = ['blower_direction', 'housing', 'board_position', 'run_case',
                            'losses', 'deflector_hole', 'material_coil_support']

# ---- funcoes copiadas do Random_for_regre.ipynb, para usar a MESMA metodologia ----
GRAFIAS_DIVERGENTES = {
    "blower_direction": {"To back": "To Back", "To front": "To Front"},
    "losses":           {"No losses": "No Losses"},
    "run_case":         {"Broil Case 2": "Broil C2", "Broil Qual": "Broil Qualitative", "Qualitative": "Broil Qualitative"},
    "housing":          {"Open - Updated": "Open"}
}


def identificar_alvos(df, independentes):
    alvos = []
    for coluna in df.columns:
        if coluna in independentes:
            continue
        if str(coluna).startswith("Unnamed:") or not str(coluna).strip():
            continue
        if df[coluna].isna().all():
            continue
        alvos.append(coluna)
    return alvos


def filtrar_alvos_por_cobertura(df, alvos, cobertura_minima):
    return [a for a in alvos if df[a].notna().mean() >= cobertura_minima]


def padronizar_categorias(df, independentes, correcoes_por_variavel):
    df = df.copy()
    for variavel, correcoes in correcoes_por_variavel.items():
        if variavel in df.columns:
            df[variavel] = df[variavel].replace(correcoes)
    for variavel in independentes:
        df[variavel] = df[variavel].fillna("Nao Informado").astype(str)
    return df


def codificar_ordinal(df, independentes, dicionario=None):
    """Se `dicionario` for passado, reusa os mesmos codigos (necessario para o
    sintetico, que precisa cair no MESMO espaco de codificacao do fold de treino)."""
    X = df[independentes].copy()
    if dicionario is None:
        dicionario = {}
        for v in independentes:
            dicionario[v] = {cat: i for i, cat in enumerate(sorted(X[v].unique()))}
    for v in independentes:
        X[v] = X[v].map(dicionario[v])
    return X, dicionario


def carregar_area(caminho):
    df = pd.read_csv(caminho)
    alvos = identificar_alvos(df, VARIAVEIS_INDEPENDENTES)
    alvos = filtrar_alvos_por_cobertura(df, alvos, COBERTURA_MINIMA_ALVO)
    tratado = df.dropna(subset=alvos).copy()
    tratado = padronizar_categorias(tratado, VARIAVEIS_INDEPENDENTES, GRAFIAS_DIVERGENTES)
    return tratado, alvos

In [ ]:
# ---- Gerador leak-free, no formato wide por area ----
# Opera DENTRO de cada fold LOO, usando so as linhas de TREINO daquele fold.
# Duas variantes, coerentes com a ablacao anterior:
#   A: configuracao inedita + interpolacao entre 2 maes do mesmo cluster termico
#   B: replica de uma linha real + ruido de medicao (regularizacao pura)
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier


def gerar_A(rng, treino_raw, alvos, n_sint):
    """treino_raw: DataFrame com as 7 colunas (ja padronizadas) + colunas de TC (alvos)."""
    if len(treino_raw) < 6:
        return None
    perfil = treino_raw[alvos]
    perfil_imp = perfil.fillna(perfil.median())
    if perfil_imp.shape[1] == 0 or perfil_imp.std().sum() == 0:
        return None
    Z = (perfil_imp - perfil_imp.mean()) / perfil_imp.std().replace(0, 1.0)
    k = min(3, max(2, len(treino_raw) // 6))
    try:
        rot = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z).labels_
    except ValueError:
        return None
    rot = pd.Series(rot, index=treino_raw.index)
    membros = {c: rot.index[rot == c].tolist() for c in np.unique(rot)}
    membros = {c: m for c, m in membros.items() if len(m) >= 2}
    if not membros:
        return None

    Xf = pd.get_dummies(treino_raw[VARIAVEIS_INDEPENDENTES].astype(str))
    clf = RandomForestClassifier(n_estimators=150, random_state=0, n_jobs=-1)
    clf.fit(Xf.values, rot.values)

    dists = {v: treino_raw[v].value_counts(normalize=True) for v in VARIAVEIS_INDEPENDENTES}
    reais = set(treino_raw[VARIAVEIS_INDEPENDENTES].astype(str).agg(' | '.join, axis=1))

    linhas, vistos, n, tent = [], set(reais), 0, 0
    while n < n_sint and tent < 4000:
        tent += 1
        cfg = {v: rng.choice(dists[v].index.to_numpy(), p=dists[v].to_numpy())
               for v in VARIAVEIS_INDEPENDENTES}
        ch = ' | '.join(str(cfg[v]) for v in VARIAVEIS_INDEPENDENTES)
        if ch in vistos:
            continue
        vistos.add(ch)
        x = pd.get_dummies(pd.DataFrame([cfg])[VARIAVEIS_INDEPENDENTES].astype(str))
        x = x.reindex(columns=Xf.columns, fill_value=0.0).astype(float)
        p = clf.predict_proba(x.values)[0]
        cands = [c for c in clf.classes_ if c in membros]
        if not cands:
            continue
        pr = np.array([p[list(clf.classes_).index(c)] for c in cands], float)
        pr = pr / pr.sum() if pr.sum() > 0 else np.ones(len(cands)) / len(cands)
        c = cands[rng.choice(len(cands), p=pr)]
        a, b = rng.choice(membros[c], size=2, replace=False)
        lam = rng.uniform(0.2, 0.8)
        y = lam * treino_raw.loc[a, alvos] + (1 - lam) * treino_raw.loc[b, alvos]
        y = y + rng.normal(0, SIGMA, len(alvos))
        linha = {**cfg, **{alvo: v for alvo, v in zip(alvos, y.values)}}
        linhas.append(linha)
        n += 1
    return pd.DataFrame(linhas) if linhas else None


def gerar_B(rng, treino_raw, alvos, n_sint):
    """Replica linha real + ruido de medicao."""
    if len(treino_raw) == 0:
        return None
    idx = treino_raw.index.to_numpy()
    linhas = []
    for _ in range(n_sint):
        mae = treino_raw.loc[idx[rng.integers(len(idx))]]
        y = mae[alvos].astype(float) + rng.normal(0, SIGMA, len(alvos))
        linha = {**{v: mae[v] for v in VARIAVEIS_INDEPENDENTES},
                 **{alvo: val for alvo, val in zip(alvos, y.values)}}
        linhas.append(linha)
    return pd.DataFrame(linhas)

### Resultado — Teste 2

Sete áreas, duas sementes cada, sempre reportando todas — inclusive a pior.

In [ ]:
# ---- Ablacao GroupKFold(5) por area, leak-free (geracao dentro de cada fold) ----
N_SINT_POR_FOLD = 10   # sintetico gerado a cada fold, proporcional ao tamanho pequeno das areas


def rodar_area(caminho, gerador, seed):
    tratado, alvos = carregar_area('../data/' + caminho)
    n = len(tratado)
    if n < 15:
        return None
    grupos = np.arange(n)
    gkf = GroupKFold(n_splits=min(N_FOLDS, n))
    rng = np.random.default_rng(seed)

    preds = np.full((n, len(alvos)), np.nan)
    Y_all = tratado[alvos].values.astype(float)

    for tr_idx, te_idx in gkf.split(np.zeros(n), groups=grupos):
        treino_raw = tratado.iloc[tr_idx]
        X_tr, dicio = codificar_ordinal(treino_raw, VARIAVEIS_INDEPENDENTES)
        Y_tr = treino_raw[alvos].values.astype(float)

        if gerador is not None:
            ds = gerador(rng, treino_raw.reset_index(drop=True).assign(
                **{v: treino_raw[v].values for v in VARIAVEIS_INDEPENDENTES}),
                alvos, N_SINT_POR_FOLD)
            if ds is not None and len(ds):
                Xs, _ = codificar_ordinal(ds, VARIAVEIS_INDEPENDENTES, dicionario=dicio)
                Ys = ds[alvos].values.astype(float)
                X_tr = pd.concat([X_tr.reset_index(drop=True), Xs.reset_index(drop=True)], ignore_index=True)
                Y_tr = np.vstack([Y_tr, Ys])

        X_te, _ = codificar_ordinal(tratado.iloc[te_idx], VARIAVEIS_INDEPENDENTES, dicionario=dicio)

        m = RandomForestRegressor(n_estimators=N_TREES, random_state=seed, n_jobs=-1)
        m.fit(X_tr.values, Y_tr)
        preds[te_idx] = m.predict(X_te.values)

    erro = preds - Y_all
    rmse = float(np.sqrt(np.nanmean(erro ** 2)))
    mae = float(np.nanmean(np.abs(erro)))
    return {'linhas': n, 'RMSE': rmse, 'MAE': mae}


SEEDS = [0, 1]
linhas_resultado = []
for area in AREAS:
    base_vals = [rodar_area(area, None, s) for s in SEEDS]
    base_vals = [v for v in base_vals if v is not None]
    if not base_vals:
        print('%-22s pulada (poucas linhas)' % area)
        continue
    base_rmse = np.mean([v['RMSE'] for v in base_vals])
    n_linhas = base_vals[0]['linhas']

    linha = {'area': area.replace('_data.csv', ''), 'linhas': n_linhas, 'so_real': round(base_rmse, 3)}
    for nome, fn in [('A_cluster', gerar_A), ('B_ruido', gerar_B)]:
        vals = [rodar_area(area, fn, s) for s in SEEDS]
        vals = [v for v in vals if v is not None]
        r = np.mean([v['RMSE'] for v in vals]) if vals else np.nan
        linha[nome] = round(r, 3)
        linha[nome + '_delta'] = round(r - base_rmse, 3) if vals else np.nan

    linhas_resultado.append(linha)
    print('%-22s linhas=%2d  so_real=%.3f  A=%.3f (%+.3f)  B=%.3f (%+.3f)'
          % (linha['area'], n_linhas, linha['so_real'],
             linha['A_cluster'], linha['A_cluster_delta'],
             linha['B_ruido'], linha['B_ruido_delta']), flush=True)

tabela_areas = pd.DataFrame(linhas_resultado).set_index('area')
print()
print(tabela_areas.to_string())
print()
print('media simples entre areas:')
print('  so_real :', round(tabela_areas['so_real'].mean(), 3))
print('  A_cluster:', round(tabela_areas['A_cluster'].mean(), 3), '  delta medio:', round(tabela_areas['A_cluster_delta'].mean(), 3))
print('  B_ruido  :', round(tabela_areas['B_ruido'].mean(), 3), '  delta medio:', round(tabela_areas['B_ruido_delta'].mean(), 3))

### Leitura

`coil_temps` piora cerca de meio grau ou mais nas duas variantes (A +1,47, B +0,50) — uma área com só 49
linhas e cobertura mais irregular, onde a validação já é instável mesmo sem
sintético. Nas outras seis, o efeito é pequeno e de sinal misto (o maior é A em `ipc_comp`, +0,54),
sem padrão que sugira uma variante ou área "boa candidata" para o sintético. Em média, as
duas variantes pioram o RMSE.


## Teste 3 — o mesmo teste, no Gradient Boosting tunado

`data/onehotGBOficial` (#196) trocou a codificação de `Grad_Boosting_Regre.ipynb`
de ordinal para one-hot e normalizou `material_coil_support`. Esse teste
replica a metodologia exata dessa versão — one-hot, `MultiOutputRegressor`
envolvendo `GradientBoostingRegressor`, mesmas correções de grafia — com os
mesmos dois geradores leak-free (A e B) do Teste 2, dentro de `GroupKFold(5)`.

**Nota à parte, não sobre o dado sintético:** a própria investigação do grupo
(`documents/hipotese_onehot_total_E18_E19.md`, protocolo `LeaveOneGroupOut` +
IC95% bootstrap, 3 sementes) concluiu **"manter ordinal — nenhum cenário
testado deu ganho real e seguro de one-hot sobre ordinal"**. Isso não é uma
contradição direta com o commit #196 — E18/E19 mediu empate estatístico, não
piora — mas significa que a troca de codificação em si não tem, hoje,
evidência própria de melhora. Registro aqui só para o grupo alinhar qual
critério prevalece; não é o objeto deste notebook.


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import GroupKFold

SEMENTE = 42
N_ARVORES = 100            # fixo, nao replica o GridSearch por area (ver ressalva)
SIGMA = 2.18
COBERTURA_MINIMA_ALVO = 0.50
N_FOLDS = 5                 # GroupKFold em vez de LeaveOneGroupOut: LOO com GBR
                             # (MultiOutputRegressor, ~10 alvos/area) e caro demais
                             # para rodar 3x por area x 2 variantes x 3 sementes.

AREAS = ['mainback_data.csv', 'coil_temps_data.csv', 'main_vent_sys_data.csv',
         'ct_knobs_data.csv', 'ipc_comp_data.csv', 'ipc_cooling_data.csv',
         'oven_hmi_data.csv']

VARIAVEIS_INDEPENDENTES = ['blower_direction', 'housing', 'board_position', 'run_case',
                            'losses', 'deflector_hole', 'material_coil_support']

# Copiado do Grad_Boosting_Regre.ipynb na branch data/onehotGBOficial (#196):
# mesma normalizacao de grafias, incluindo a nova de material_coil_support, e
# codificacao ONE-HOT em vez de ordinal (a mudanca que aquele commit fez).
GRAFIAS_DIVERGENTES = {
    "blower_direction": {"To back": "To Back", "To front": "To Front"},
    "losses":           {"No losses": "No Losses"},
    "run_case":         {"Broil Case 2": "Broil C2", "Broil Qual": "Broil Qualitative", "Qualitative": "Broil Qualitative"},
    "housing":          {"Open - Updated": "Open"},
    "material_coil_support": {"Aluminium Plate": "Aluminium "},
}


def identificar_alvos(df, independentes):
    alvos = []
    for coluna in df.columns:
        if coluna in independentes:
            continue
        if str(coluna).startswith("Unnamed:") or not str(coluna).strip():
            continue
        if df[coluna].isna().all():
            continue
        alvos.append(coluna)
    return alvos


def filtrar_alvos_por_cobertura(df, alvos, cobertura_minima):
    return [a for a in alvos if df[a].notna().mean() >= cobertura_minima]


def padronizar_categorias(df, independentes, correcoes_por_variavel):
    df = df.copy()
    for variavel, correcoes in correcoes_por_variavel.items():
        if variavel in df.columns:
            df[variavel] = df[variavel].replace(correcoes)
    for variavel in independentes:
        df[variavel] = df[variavel].fillna("Nao Informado").astype(str)
    return df


def codificar_onehot(df, independentes, colunas_referencia=None):
    """Se `colunas_referencia` for passado, reindexa para o MESMO espaco de
    colunas do fold de treino (necessario para o sintetico e para o holdout)."""
    X = pd.get_dummies(df[independentes].astype(str), columns=independentes).astype(int)
    if colunas_referencia is not None:
        X = X.reindex(columns=colunas_referencia, fill_value=0)
    return X


def carregar_area(caminho):
    df = pd.read_csv(caminho)
    alvos = identificar_alvos(df, VARIAVEIS_INDEPENDENTES)
    alvos = filtrar_alvos_por_cobertura(df, alvos, COBERTURA_MINIMA_ALVO)
    tratado = df.dropna(subset=alvos).copy()
    tratado = padronizar_categorias(tratado, VARIAVEIS_INDEPENDENTES, GRAFIAS_DIVERGENTES)
    return tratado.reset_index(drop=True), alvos

In [ ]:
# ---- Geradores leak-free (mesma logica de A/B do teste com RandomForest) ----
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier


def gerar_A(rng, treino_raw, alvos, n_sint):
    if len(treino_raw) < 6:
        return None
    perfil = treino_raw[alvos]
    perfil_imp = perfil.fillna(perfil.median())
    if perfil_imp.shape[1] == 0 or perfil_imp.std().sum() == 0:
        return None
    Z = (perfil_imp - perfil_imp.mean()) / perfil_imp.std().replace(0, 1.0)
    k = min(3, max(2, len(treino_raw) // 6))
    try:
        rot = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z).labels_
    except ValueError:
        return None
    rot = pd.Series(rot, index=treino_raw.index)
    membros = {c: rot.index[rot == c].tolist() for c in np.unique(rot)}
    membros = {c: m for c, m in membros.items() if len(m) >= 2}
    if not membros:
        return None

    Xf = pd.get_dummies(treino_raw[VARIAVEIS_INDEPENDENTES].astype(str))
    clf = RandomForestClassifier(n_estimators=150, random_state=0, n_jobs=-1)
    clf.fit(Xf.values, rot.values)

    dists = {v: treino_raw[v].value_counts(normalize=True) for v in VARIAVEIS_INDEPENDENTES}
    reais = set(treino_raw[VARIAVEIS_INDEPENDENTES].astype(str).agg(' | '.join, axis=1))

    linhas, vistos, n, tent = [], set(reais), 0, 0
    while n < n_sint and tent < 4000:
        tent += 1
        cfg = {v: rng.choice(dists[v].index.to_numpy(), p=dists[v].to_numpy())
               for v in VARIAVEIS_INDEPENDENTES}
        ch = ' | '.join(str(cfg[v]) for v in VARIAVEIS_INDEPENDENTES)
        if ch in vistos:
            continue
        vistos.add(ch)
        x = pd.get_dummies(pd.DataFrame([cfg])[VARIAVEIS_INDEPENDENTES].astype(str))
        x = x.reindex(columns=Xf.columns, fill_value=0.0).astype(float)
        p = clf.predict_proba(x.values)[0]
        cands = [c for c in clf.classes_ if c in membros]
        if not cands:
            continue
        pr = np.array([p[list(clf.classes_).index(c)] for c in cands], float)
        pr = pr / pr.sum() if pr.sum() > 0 else np.ones(len(cands)) / len(cands)
        c = cands[rng.choice(len(cands), p=pr)]
        a, b = rng.choice(membros[c], size=2, replace=False)
        lam = rng.uniform(0.2, 0.8)
        y = lam * treino_raw.loc[a, alvos] + (1 - lam) * treino_raw.loc[b, alvos]
        y = y + rng.normal(0, SIGMA, len(alvos))
        linhas.append({**cfg, **{alvo: v for alvo, v in zip(alvos, y.values)}})
        n += 1
    return pd.DataFrame(linhas) if linhas else None


def gerar_B(rng, treino_raw, alvos, n_sint):
    if len(treino_raw) == 0:
        return None
    idx = treino_raw.index.to_numpy()
    linhas = []
    for _ in range(n_sint):
        mae = treino_raw.loc[idx[rng.integers(len(idx))]]
        y = mae[alvos].astype(float) + rng.normal(0, SIGMA, len(alvos))
        linhas.append({**{v: mae[v] for v in VARIAVEIS_INDEPENDENTES},
                       **{alvo: val for alvo, val in zip(alvos, y.values)}})
    return pd.DataFrame(linhas)

### Resultado — Teste 3

In [ ]:
# ---- Ablacao GroupKFold(5) por area, leak-free, Gradient Boosting one-hot ----
N_SINT_POR_FOLD = 15


def rodar_area_gb(caminho, gerador, seed):
    tratado, alvos = carregar_area('../data/' + caminho)
    n = len(tratado)
    if n < 15:
        return None
    grupos = np.arange(n)
    gkf = GroupKFold(n_splits=min(N_FOLDS, n))
    rng = np.random.default_rng(seed)

    preds = np.full((n, len(alvos)), np.nan)
    Y_all = tratado[alvos].values.astype(float)

    for tr_idx, te_idx in gkf.split(np.zeros(n), groups=grupos):
        treino_raw = tratado.iloc[tr_idx]
        X_tr = codificar_onehot(treino_raw, VARIAVEIS_INDEPENDENTES)
        Y_tr = treino_raw[alvos].values.astype(float)

        if gerador is not None:
            ds = gerador(rng, treino_raw, alvos, N_SINT_POR_FOLD)
            if ds is not None and len(ds):
                Xs = codificar_onehot(ds, VARIAVEIS_INDEPENDENTES, colunas_referencia=X_tr.columns)
                Ys = ds[alvos].values.astype(float)
                X_tr = pd.concat([X_tr.reset_index(drop=True), Xs.reset_index(drop=True)], ignore_index=True)
                Y_tr = np.vstack([Y_tr, Ys])

        X_te = codificar_onehot(tratado.iloc[te_idx], VARIAVEIS_INDEPENDENTES, colunas_referencia=X_tr.columns)

        m = MultiOutputRegressor(
            GradientBoostingRegressor(n_estimators=N_ARVORES, random_state=seed), n_jobs=-1)
        m.fit(X_tr.values, Y_tr)
        preds[te_idx] = m.predict(X_te.values)

    erro = preds - Y_all
    rmse = float(np.sqrt(np.nanmean(erro ** 2)))
    return {'linhas': n, 'RMSE': rmse}


SEEDS_GB = [0, 1]   # 2 sementes: GBR e mais caro que RF, GroupKFold(5) ja reduz o custo
linhas_gb = []
for area in AREAS:
    base_vals = [rodar_area_gb(area, None, s) for s in SEEDS_GB]
    base_vals = [v for v in base_vals if v is not None]
    if not base_vals:
        print('%-22s pulada (poucas linhas)' % area)
        continue
    base_rmse = np.mean([v['RMSE'] for v in base_vals])
    n_linhas = base_vals[0]['linhas']

    linha = {'area': area.replace('_data.csv', ''), 'linhas': n_linhas, 'so_real': round(base_rmse, 3)}
    for nome, fn in [('A_cluster', gerar_A), ('B_ruido', gerar_B)]:
        vals = [rodar_area_gb(area, fn, s) for s in SEEDS_GB]
        vals = [v for v in vals if v is not None]
        r = np.mean([v['RMSE'] for v in vals]) if vals else np.nan
        linha[nome] = round(r, 3)
        linha[nome + '_delta'] = round(r - base_rmse, 3) if vals else np.nan

    linhas_gb.append(linha)
    print('%-22s linhas=%2d  so_real=%.3f  A=%.3f (%+.3f)  B=%.3f (%+.3f)'
          % (linha['area'], n_linhas, linha['so_real'],
             linha['A_cluster'], linha['A_cluster_delta'],
             linha['B_ruido'], linha['B_ruido_delta']), flush=True)

tabela_gb = pd.DataFrame(linhas_gb).set_index('area')
print()
print(tabela_gb.to_string())
print()
print('media simples entre areas:')
print('  so_real  :', round(tabela_gb['so_real'].mean(), 3))
print('  A_cluster:', round(tabela_gb['A_cluster'].mean(), 3), '  delta medio:', round(tabela_gb['A_cluster_delta'].mean(), 3))
print('  B_ruido  :', round(tabela_gb['B_ruido'].mean(), 3), '  delta medio:', round(tabela_gb['B_ruido_delta'].mean(), 3))

### Leitura

Mesmo padrão dos Testes 1 e 2: A piora (`coil_temps` +0,858 °C, `oven_hmi`
+0,498 °C), B fica neutro (delta médio −0,035 °C, sinal misto entre áreas,
consistente com ruído). O algoritmo mudou — RandomForest para Gradient
Boosting, ordinal para one-hot — a conclusão sobre o dado sintético não.


## Achado independente: as features escondidas, não as linhas

Em paralelo, o commit `3bb91c6` (Lucas Andrade, #186) removeu um notebook
exploratório — `Random_for_regre_reasons.ipynb` — que testava decodificar
`reason` em colunas, por um método diferente do usado neste notebook.

O achado principal de lá, antes de ser removido: **o piso de erro irredutível
caía 44,6% em média (2,58 °C → 1,51 °C por área)** ao adicionar essas colunas
— quase idêntico aos 45% medidos de forma independente no Teste 1 (7,41 → 4,09
°C, no dataset agregado). Duas pessoas, dois métodos de extração, mesmo
resultado. Isso não é uma coincidência de arredondamento — é o mesmo
fenômeno físico sendo medido duas vezes: runs com a mesma entrada nas 7
colunas atuais, mas resultados diferentes, porque a variável que as distingue
(velocidade do soprador, presença de cooktop, etc.) está escrita em texto e
nunca virou coluna.

A decisão de não adotar não foi "não funciona" — foi que o MAE/R² no conjunto
de teste de 9 linhas por área ficou estatisticamente empatado, e o piso (que
não depende do split) não é uma métrica de teste que se possa citar sozinha
como prova de ganho no holdout. Ambos os pontos são verdadeiros ao mesmo
tempo: o piso caiu de verdade, e o teste de 9 linhas não teve poder
estatístico para confirmar isso no MAE.


## Conclusão

| Teste | Método | Resultado |
|---|---|---|
| 1 | Dataset agregado, 3 variantes de geração, harness leak-free | Nenhuma melhora |
| 2 | Formato wide por área, metodologia do `Random_for_regre.ipynb`, leak-free | Nenhuma melhora, `coil_temps` piora bastante |
| 3 | Formato wide por área, Gradient Boosting one-hot tunado (#196), leak-free | Mesmo padrão: A piora, B neutro |
| — | Extração independente de `reason` (Lucas, #186) | Piso cai 44,6% — corrobora o enriquecimento do Teste 1 |

**O gargalo é feature, não volume.** Os Testes 1, 2 e 3 fecham a pergunta de
gerar mais linhas com as 7 variáveis atuais em três algoritmos diferentes
(RandomForest agregado, RandomForest por área, Gradient Boosting por área):
não ajuda em nenhum. O achado independente do Lucas, junto com o próprio
Teste 1, aponta o caminho que tem evidência de funcionar: **decodificar
`reason` em colunas**.

**O que fica pendente, fora do escopo deste notebook:** levar essa extração
aos três notebooks de modelo publicados e medir o ganho com a validação LOO
que eles já usam. Isso muda os modelos de fato — decisão do grupo, feita
deliberadamente fora deste registro.
